In [ ]:
import glob
import pandas as pd

file_path = '데이터분석\Data\소상공인시장진흥공단_상가(상권)정보_서울_202512.csv'
shop = pd.read_csv(file_path)
shop.columns

In [14]:
shop['상권업종중분류명'].unique()

array(['기타 간이', '고용 알선', '비알코올 ', '유원지·오락', '시계·귀금속 소매', '부동산 서비스',
       '섬유·의복·신발 소매', '주점', '가구 소매', '기타 사업 서비스', '스포츠 서비스', '한식',
       '의약·화장품 소매', '본사·경영 컨설팅', '자동차 수리·세차', '오락용품 소매', '철물·건설자재 소매',
       '가전·통신 소매', '회계·세무', '의원', '통신장비 수리', '여행사·보조', '담배 소매', '장식품 소매',
       '연료 소매', '이용·미용', '일반 숙박', '법무관련 ', '모터사이클 소매', '중식', '기술 서비스',
       '일반 교육', '산업용품 대여', '가정용품 대여', '세탁', '종합 소매', '전문 디자인', '일식',
       '기타 전문 과학', '기타 교육', '식료품 소매', '병원', '기타 숙박', '중고 상품 소매', '교육 지원',
       '시설관리', '청소·방제', '사진 촬영', '기타 생활용품 소매', '인쇄·제품제작', '안경·정밀기기 소매',
       '광고', '욕탕·신체관리', '기타 상품 소매', '음료 소매', '기타 가정용품 수리', '수의',
       '모터사이클 수리', '가전제품 수리', '동남아시아', '서양식', '도서관·사적지', '식물 소매',
       '구내식당·뷔페', '컴퓨터 수리', '시장 조사', '사무 지원', '기타 개인', '장례식장 ',
       '자동차 부품 소매', '운송장비 대여', '애완동물·용품 소매', '기타 보건', '조경·유지', '기타 외국'],
      dtype=object)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree
from lightgbm import LGBMRegressor

# ---------------------------
# 0. 데이터 준비
# ---------------------------
# shop: 기존 데이터프레임
shop = shop.dropna(subset=['위도','경도','상권업종중분류명']).reset_index(drop=True)

# 좌표 → 라디안 변환
coords = np.radians(shop[['위도','경도']].values)

# BallTree 생성
tree = BallTree(coords, metric='haversine')

# 반경 설정 (500m)
radius = 0.5 / 6371


# ---------------------------
# 1. 전체 점포 밀집도
# ---------------------------
store_counts = tree.query_radius(coords, r=radius, count_only=True)
shop['store_density_500m'] = store_counts


# ---------------------------
# 2. 동일 업종 밀집도
# ---------------------------
target_category = '중식'

cafe_mask = (shop['상권업종중분류명'] == target_category)
cafe_coords = np.radians(shop.loc[cafe_mask, ['위도','경도']].values)

cafe_tree = BallTree(cafe_coords, metric='haversine')

cafe_counts = cafe_tree.query_radius(coords, r=radius, count_only=True)
shop['cafe_density_500m'] = cafe_counts


# ---------------------------
# 3. 경쟁 강도
# ---------------------------
shop['competition_ratio'] = (
    shop['cafe_density_500m'] / shop['store_density_500m']
).replace([np.inf, -np.inf], 0).fillna(0)


# ---------------------------
# 4. 상권 다양성
# ---------------------------
indices = tree.query_radius(coords, r=radius)

diversity = []
for idx_list in indices:
    categories = shop.iloc[idx_list]['상권업종중분류명']
    diversity.append(categories.nunique())

shop['category_diversity_500m'] = diversity


# ---------------------------
# 5. Feature 정리
# ---------------------------
features = [
    'store_density_500m',
    'cafe_density_500m',
    'competition_ratio',
    'category_diversity_500m',
    '위도',
    '경도'
]

X = shop[features]


# ---------------------------
# 6. Score 생성 (라벨 대신)
# ---------------------------
shop['score'] = (
    0.4 * shop['store_density_500m']
    - 0.3 * shop['competition_ratio']
    + 0.3 * shop['category_diversity_500m']
)

y = shop['score']


# ---------------------------
# 7. 모델 학습
# ---------------------------
model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    random_state=42
)

model.fit(X, y)


# ---------------------------
# 8. 후보 위치 (Grid 생성)
# ---------------------------
lat_min, lat_max = shop['위도'].min(), shop['위도'].max()
lon_min, lon_max = shop['경도'].min(), shop['경도'].max()

lat_grid = np.linspace(lat_min, lat_max, 50)
lon_grid = np.linspace(lon_min, lon_max, 50)

grid = np.array(np.meshgrid(lat_grid, lon_grid)).T.reshape(-1, 2)

grid_df = pd.DataFrame(grid, columns=['위도','경도'])

grid_coords = np.radians(grid)


# ---------------------------
# 9. Grid Feature 생성
# ---------------------------
# 전체 밀집도
grid_store_counts = tree.query_radius(grid_coords, r=radius, count_only=True)

# 카페 밀집도
grid_cafe_counts = cafe_tree.query_radius(grid_coords, r=radius, count_only=True)

grid_df['store_density_500m'] = grid_store_counts
grid_df['cafe_density_500m'] = grid_cafe_counts

# 경쟁 강도
grid_df['competition_ratio'] = (
    grid_df['cafe_density_500m'] / grid_df['store_density_500m']
).replace([np.inf, -np.inf], 0).fillna(0)

# 다양성 (주의: 느림)
grid_indices = tree.query_radius(grid_coords, r=radius)

grid_diversity = []
for idx_list in grid_indices:
    categories = shop.iloc[idx_list]['상권업종중분류명']
    grid_diversity.append(categories.nunique())

grid_df['category_diversity_500m'] = grid_diversity


# ---------------------------
# 10. 예측 및 추천
# ---------------------------
grid_X = grid_df[features]

grid_df['pred_score'] = model.predict(grid_X)

# Top 10 추천
top_k = grid_df.sort_values('pred_score', ascending=False).head(10)
print(top_k)